In [44]:
import pandas as pd
import gdown
import os
import numpy as np

In [45]:
# dict_plikow = {
#     'towar': 'https://drive.google.com/file/d/1BnYuwCY4A7fzb5Vg36r6vpCbRhM_acDZ/view?usp=drive_link',
#     'pozdok': 'https://drive.google.com/file/d/1q9x3lCAb9u0BPqbEzGJLhdkESDon33Sl/view?usp=drive_link',
#     'dok':'https://drive.google.com/file/d/1ONZ1yvmTAsGq1P9FieuZvyRINUNHj1dk/view?usp=drive_link',
#     'asort': 'https://drive.google.com/file/d/15vZYClznfSQDGO_qS0gPJ3J9Vo5ZKzsS/view?usp=sharing'
# }



In [46]:
# force_redownload = False  # ustaw True, gdy chcesz odświeżyć dane
# file_path = "dane"

# for key, url in dict_plikow.items():
#     file_path = f"dane/{key}.parquet"
#     if os.path.exists(file_path) and not force_redownload:
#         print(f"Plik {file_path} już istnieje - pomijam.")
#         continue
#     gdown.download(url, file_path, quiet=False)

#  DOK

In [47]:
pozdok = pd.read_parquet('dane/pozdok.parquet', engine='pyarrow')


In [48]:
pozd_cols = ['DokId', 'TowId', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPrzedRab', 'CenaPoRab', 'Wartosc', 'CenaDet']

In [49]:

pozdok = pozdok[pozd_cols].copy()

In [50]:

asort = pd.read_parquet('dane/asort.parquet')

In [51]:
asort_cols = ['AsId', 'Nazwa']
asort= asort[asort_cols].copy()


In [52]:
asort.sample(10)

,AsId,Nazwa
47,88,MLEKO /nabiał
19,30,KREMY DO SMAROWANIA /dodatki deserowe
123,247,OPAKOWANIA /przemysłowe
281,230,WAFLE I PODKŁADY TORTOWE
325,235,MARGARYNY UNIVERSALNE /tłuszcze
9,10,CHUSTECZKI HIGIENICZNE /papierniczo hig
143,278,12 PRZEMYSŁOWE KIOSK
314,202,OWOCOWE /przetwory owocowe
135,264,Admat Frytki
242,115,WARZYWA


## Towar

In [53]:
towar = pd.read_parquet('dane/towar.parquet')

In [54]:
tow_cols = ['TowId', 'AsId',  'Nazwa', 'Kod', 'Opis1', 'Producent', 'Marza', 'Stawka', 'Aktywny']
towar = towar[tow_cols].copy()

## DOK

In [55]:

dok = pd.read_parquet('dane/dok.parquet')

In [56]:
dok_cols = ['DokId', 'Data', 'KolejnyWDniu', 'NrDok', 'TypDok', 'Aktywny', 'Razem', 'DoZaplaty', 'Zaplacono']
dok = dok[dok_cols].copy()


In [57]:
dok['Data'] = pd.to_datetime(dok['Data'], errors='coerce').copy()


## Usuwam te rekordy które w NrDok maja "NaN"

In [58]:
dok = dok[dok['NrDok'].notna()]

#  POZDOK

In [59]:
pozdok = pozdok.drop_duplicates()

## Łacze dok oraz pozdok 

In [60]:
pozdok_dok = pd.merge(pozdok, dok, on="DokId", how="left")

In [61]:
pozdok_dok

,DokId,TowId,TypPoz,IloscPlus,IloscMinus,CenaPrzedRab,CenaPoRab,Wartosc,CenaDet,Data,KolejnyWDniu,NrDok,TypDok,Aktywny,Razem,DoZaplaty,Zaplacono
0,41775,19623,0,1.000,0.0,917.1500,917.1500,917.1500,745.6504,2018-09-14,519.0,1/09/L,33.0,1.0,1128.09,1128.09,1128.09
1,131173,23638,0,1.000,0.0,112.5000,112.5000,112.5000,0.0000,2019-01-07,178.0,FV/19/4,33.0,1.0,138.38,138.38,138.38
2,142162,23638,0,1.000,0.0,75.0000,75.0000,75.0000,0.0000,2019-01-21,1.0,FV/19/7,33.0,1.0,92.25,92.25,92.25
3,159723,23638,0,1.000,0.0,7956.8100,7956.8100,7956.8100,0.0000,2019-02-11,305.0,FV/19/9,33.0,1.0,9786.88,9786.88,9786.88
4,162945,25390,0,1.000,0.0,100.0000,100.0000,100.0000,100.0407,2019-02-22,2.0,FV/19/10,33.0,1.0,123.00,123.00,123.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3812452,2237660,81803,4,1.000,0.0,5.2764,5.2764,5.2764,5.2764,2026-01-31,947.0,DF/PAR/260131/12424/2,21.0,1.0,96.23,96.23,96.23
3812453,2237660,81638,4,1.000,0.0,0.5000,0.5000,0.5000,0.5000,2026-01-31,947.0,DF/PAR/260131/12424/2,21.0,1.0,96.23,96.23,96.23
3812454,2237660,52905,4,1.000,0.0,2.4309,2.4309,2.4309,2.4309,2026-01-31,947.0,DF/PAR/260131/12424/2,21.0,1.0,96.23,96.23,96.23
3812455,2237660,80379,4,1.000,0.0,15.4390,15.4390,15.4390,15.4390,2026-01-31,947.0,DF/PAR/260131/12424/2,21.0,1.0,96.23,96.23,96.23


In [62]:
pozdok_dok=pozdok_dok.rename(columns={'RabatProc_x': 'RabatProc'}).copy()

In [63]:
asort = asort.rename(columns={'Nazwa': 'Nazwa_asort'})

In [64]:
towar = towar.rename(columns={'Nazwa': 'Nazwa_towar'})

In [65]:
# towar = towar.drop_duplicates()

In [66]:
towar['Kod'] = towar['Kod'].fillna(0)
towar['Kod']=towar['Kod'].astype('int')


## Łacze dataframes  "towar" oraz "asort" powstaje "towar_asort"

In [67]:
towar_asort = pd.merge(towar, asort, on="AsId", how="left")

In [68]:
towar_asort.sample(3)

,TowId,AsId,Nazwa_towar,Kod,Opis1,Producent,Marza,Stawka,Aktywny,Nazwa_asort
1904,21269,293,TYT PONIATOWSKI40G VANILLA,5712341060449,nan,NaN,0.0,2300,0,02 TYTOŃ W KIOSK
30152,80248,106,CAPUCINO MALINA KOKOS ZERO 20G MOKATE,5900649091009,nan,NaN,0.2,2300,1,KAWY
21797,1939,221,"CUKIERKI HALLS EXTRA STRONG 33,5G MONDELEZ",50985098,SLODYCZE,328.0,0.2,2300,1,CUKIERKI


## Łacze dataframes  "pozdok_dok" oraz "towar_asort" powstaje "pozdok_dok_towar_asort"

In [69]:
df = pd.merge(pozdok_dok, towar_asort, on="TowId", how="left")


In [70]:
df.Data.min(), df.Data.max()

(Timestamp('2006-01-27 00:00:00'), Timestamp('2026-02-02 00:00:00'))

In [71]:
df = df[df['Data'].between('2023-01-01', '2025-12-31')].copy()

In [72]:
df = df.dropna(subset=['TypDok'])

In [73]:
df = df.rename(columns={'TowId': 'SKU'})

In [74]:
df = df[df['Nazwa_towar'].notna()]

In [75]:
df = df[df['Nazwa_towar'] != 'USŁUGA MARKETINGOWA']

In [76]:
df[df.Aktywny_x!=df.Aktywny_y]

,DokId,SKU,TypPoz,IloscPlus,IloscMinus,CenaPrzedRab,CenaPoRab,Wartosc,CenaDet,Data,...,Zaplacono,AsId,Nazwa_towar,Kod,Opis1,Producent,Marza,Stawka,Aktywny_y,Nazwa_asort
1146,1166195,37370,4,1.0,0.0,4.7886,4.7886,4.7886,4.7886,2023-01-02,...,12.69,98.0,"NAPÓJ GAZ BUT SCHWEPPES 0,9L TONIC",5.902860e+12,nan,NaN,0.20,2300.0,0.0,NAPOJE GAZOWANE
1152,1166198,54867,4,1.0,0.0,2.2900,2.2900,2.2900,2.2900,2023-01-02,...,11.87,217.0,BATON PIERNIKOWY MARCEPAN 47G KOPERNIK,5.900056e+12,nan,NaN,0.20,500.0,0.0,BATONY I WAFELKI
1160,1166201,7427,4,1.0,0.0,3.4900,3.4900,3.4900,3.4900,2023-01-02,...,10.27,87.0,KEFIR KUB LUKSUSOWY 2% NATURALNY 400G ŁÓDŹ,5.900771e+12,NABIAL,241.0,0.25,500.0,0.0,KEFIRY I MAŚLANKI /nabiał
1163,1166203,55334,4,1.0,0.0,6.9900,6.9900,6.9900,6.9900,2023-01-02,...,17.15,233.0,MARG FLORA 225G UPFIELD,8.719200e+12,nan,NaN,0.20,0.0,0.0,MARGARYNY DO SMAROWANIA /tłuszcze
1189,1166214,780,4,1.0,0.0,4.6204,4.6204,4.6204,4.6204,2023-01-02,...,15.52,224.0,GUMY DRAŻETKI ORBIT SPEARMINT 25SZT 35G WRIGLEY,4.009900e+12,SPOZYWCZ,580.0,0.20,800.0,0.0,GUMY DO ŻUCIA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3713500,2211813,52428,4,1.0,0.0,7.3981,7.3981,7.3981,7.3981,2025-12-31,...,43.67,40.0,MAJONEZ HELMANS BABUNI 405ML UNILEVER,8.720182e+12,nan,NaN,0.20,800.0,0.0,MAJONEZY /dodatki spożywcze
3713751,2211939,5934,6,0.0,0.0,4.9000,6.4959,0.0000,5.2764,2025-12-31,...,0.00,98.0,"NAPÓJ GAZ BUT MAUNTAM DEW 1,5L PEPSI",5.900497e+12,NAPOJE,173.0,0.00,2300.0,0.0,NAPOJE GAZOWANE
3713785,2211939,18743,6,0.0,0.0,4.1000,5.7048,0.0000,3.8000,2025-12-31,...,0.00,244.0,"ZUPA BARSZCZ CZERWONY 1,1L DAWTONA",5.901713e+12,nan,NaN,0.25,800.0,0.0,Domyślny DO WERYFIKACJI + PRZECENY
3713823,2211940,5934,6,0.0,0.0,4.9000,6.4959,0.0000,5.2764,2025-12-31,...,0.00,98.0,"NAPÓJ GAZ BUT MAUNTAM DEW 1,5L PEPSI",5.900497e+12,NAPOJE,173.0,0.00,2300.0,0.0,NAPOJE GAZOWANE


## Analiza czasowa

- sprzedaż w czasie (miesięcznie/tygodniowo) — Data
- które miesiące/dni mają największy obrót

In [77]:
df['Data'] = pd.to_datetime(df['Data'])

# Kolumny czasowe
df['Rok'] = df['Data'].dt.year
df['Miesiac'] = df['Data'].dt.month
df['RokMiesiac'] = df['Data'].dt.to_period('M')
df['DzienTygodnia'] = df['Data'].dt.dayofweek  # 0=pon, 6=nd

In [78]:
lista_typdok = df['TypDok'].value_counts().index.tolist()

In [79]:

# for l in lista_typdok:
#     df_temp = df[df['TypDok'] == l]
#     print(f"\n--- TypDok: {l} ---")
#     display(df_temp.sample(1))
 

In [80]:
niedziele_handlowe = pd.to_datetime([
    # 2022
    '2022-01-30', '2022-04-10', '2022-04-24', '2022-06-26', '2022-08-28', '2022-12-11', '2022-12-18',
    # 2023
    '2023-01-29', '2023-04-02', '2023-04-30', '2023-06-25', '2023-08-27', '2023-12-10', '2023-12-17',
    # 2024
    '2024-01-28', '2024-03-24', '2024-04-28', '2024-06-30', '2024-08-25', '2024-12-15', '2024-12-22',
    # 2025
    '2025-01-26', '2025-04-13', '2025-04-27', '2025-06-29', '2025-08-31', '2025-12-14', '2025-12-21'
])

In [81]:
long_weekend_dates = [
    # ===== 2022 =====
    "2022-01-01", "2022-01-02",
    "2022-01-06", "2022-01-07", "2022-01-08", "2022-01-09",
    "2022-04-16", "2022-04-17", "2022-04-18",
    "2022-06-16", "2022-06-17", "2022-06-18", "2022-06-19",
    "2022-10-29", "2022-10-30", "2022-10-31", "2022-11-01",
    "2022-11-11", "2022-11-12", "2022-11-13",
    # ===== 2023 =====
    "2023-04-08", "2023-04-09", "2023-04-10",
    "2023-06-08", "2023-06-09", "2023-06-10", "2023-06-11",
    "2023-08-12", "2023-08-13", "2023-08-14", "2023-08-15",
    "2023-10-28", "2023-10-29", "2023-10-30", "2023-10-31", "2023-11-01",
    # ===== 2024 =====
    "2024-01-01",
    "2024-03-30", "2024-03-31", "2024-04-01",
    "2024-05-30", "2024-05-31", "2024-06-01", "2024-06-02",
    "2024-08-15", "2024-08-16", "2024-08-17", "2024-08-18",
    "2024-11-01", "2024-11-02", "2024-11-03",
    "2024-11-09", "2024-11-10", "2024-11-11",
    # ===== 2025 =====
    "2025-04-19", "2025-04-20", "2025-04-21",
    "2025-06-19", "2025-06-20", "2025-06-21", "2025-06-22",
    "2025-08-15", "2025-08-16", "2025-08-17",
]

In [ ]:

df['Data'] = pd.to_datetime(df['Data'], errors='coerce')

df['IloscPlus']   = pd.to_numeric(df['IloscPlus'],   errors='coerce').fillna(0)
df['IloscMinus']  = pd.to_numeric(df['IloscMinus'],  errors='coerce').fillna(0)
df = df.dropna(subset=['Data'])
df['Data'] = df['Data'].dt.floor('D')
df['czy_remanent'] = df['TypDok'] == 16


df = df.dropna(subset=['NrDok'])
df = df.rename(columns={'TowId_Log': 'SKU'})


ip_im_mask = df['TypDok'].isin([16, 26, 78, 88])
# df.loc[ip_im_mask, 'ilosc_netto'] = (
#     df.loc[ip_im_mask, 'IloscPlus'] - df.loc[ip_im_mask, 'IloscMinus']
# )
# TypDok 21 (sprzedaż) — towar wychodzi, ilosc_netto ujemna
# sprzedaz_mask = df['TypDok'] == 21
# df.loc[sprzedaz_mask, 'ilosc_netto'] = -df.loc[sprzedaz_mask, 'IloscMinus']
# df['Ruch'] = df['ilosc_netto']

# Dni tygodnia
df['nazwa_dnia'] = df['Data'].dt.dayofweek.map({
    0: 'Poniedziałek', 1: 'Wtorek', 2: 'Środa',
    3: 'Czwartek', 4: 'Piątek', 5: 'Sobota', 6: 'Niedziela'
})

# Flagi specjalnych dni
df['dlugi_weekend']    = df['Data'].isin(long_weekend_dates)
df['niedziela_handlowa'] = df['Data'].isin(niedziele_handlowe)

# Kategoria dnia
def kategoryzuj(row):
    if row['niedziela_handlowa']:
        return 'Niedziela handlowa'
    if row['dlugi_weekend']:
        return 'Długi weekend'
    return row['nazwa_dnia']

df['kategoria_dnia'] = df.apply(kategoryzuj, axis=1)

# Flagi specjalnych dni
df['dlugi_weekend']    = df['Data'].isin(long_weekend_dates)
df['niedziela_handlowa'] = df['Data'].isin(niedziele_handlowe)

# Kategoria dnia
def kategoryzuj(row):
    if row['niedziela_handlowa']:
        return 'Niedziela handlowa'
    if row['dlugi_weekend']:
        return 'Długi weekend'
    return row['nazwa_dnia']

df['kategoria_dnia'] = df.apply(kategoryzuj, axis=1)

# Słownik nazw
etykiety = (
    df[df['Nazwa_towar'].notna()][['SKU', 'Nazwa_towar']]
    .drop_duplicates('SKU')
    .set_index('SKU')
)
wszystkie_ids = pd.DataFrame({'SKU': df['SKU'].unique()})
wszystkie_ids = wszystkie_ids.merge(etykiety.reset_index(), on='SKU', how='left')
wszystkie_ids['Nazwa_towar'] = wszystkie_ids['Nazwa_towar'].fillna(
    wszystkie_ids['SKU'].astype('int').apply(lambda x: f'[ID {x}]')
)
wszystkie_ids = wszystkie_ids.sort_values('Nazwa_towar').reset_index(drop=True)

# Uzupełnij Nazwa w df
df['Nazwa_towar'] = df['SKU'].map(wszystkie_ids.set_index('SKU')['Nazwa_towar'])
df['Nazwa_towar'] = df['Nazwa_towar'].fillna(df['SKU'].astype('int').apply(lambda x: f'[ID {x}]'))

print(f"Wierszy: {len(df):,}  |  SKU: {df['SKU'].nunique():,}  |  Zakres: {df['Data'].min().date()} – {df['Data'].max().date()}")
df.sample(3)

Wierszy: 3,693,844  |  SKU: 21,865  |  Zakres: 2023-01-01 – 2025-12-31


,DokId,SKU,TypPoz,IloscPlus,IloscMinus,CenaPrzedRab,CenaPoRab,Wartosc,CenaDet,Data,...,Miesiac,RokMiesiac,DzienTygodnia,czy_remanent,ilosc_netto,Ruch,nazwa_dnia,dlugi_weekend,niedziela_handlowa,kategoria_dnia
1078965,1452040,294,4,1.000,0.0,5.4900,5.4900,5.4900,5.4900,2023-11-20,...,11,2023-11,0,False,-0.0,-0.0,Poniedziałek,False,False,Poniedziałek
502352,1300773,49624,4,0.596,0.0,6.9900,6.9900,4.1700,6.9900,2023-05-27,...,5,2023-05,5,False,-0.0,-0.0,Sobota,False,False,Sobota
1064187,1448309,38960,4,1.000,0.0,0.5691,0.5691,0.5691,0.5691,2023-11-15,...,11,2023-11,2,False,-0.0,-0.0,Środa,False,False,Środa


In [40]:
df.sample(3)

,DokId,SKU,TypPoz,IloscPlus,IloscMinus,CenaPrzedRab,CenaPoRab,Wartosc,CenaDet,Data,...,Miesiac,RokMiesiac,DzienTygodnia,czy_remanent,ilosc_netto,Ruch,nazwa_dnia,dlugi_weekend,niedziela_handlowa,kategoria_dnia
3419197,2134704,78750,4,1.0,0.0,16.252,16.252,16.252,16.2520,2025-09-30,...,9,2025-09,1,False,-0.0,-0.0,Wtorek,False,False,Wtorek
3137618,2048522,54776,6,54.0,0.0,3.490,3.800,205.200,5.7048,2025-07-01,...,7,2025-07,1,False,NaN,NaN,Wtorek,False,False,Wtorek
3002410,2011673,10022,4,1.0,0.0,2.187,2.187,2.187,2.1870,2025-05-17,...,5,2025-05,5,False,-0.0,-0.0,Sobota,False,False,Sobota


In [41]:
df['TypPoz'] = df['TypPoz'].astype('int')
df['IloscPlus'] = df['IloscPlus'].astype('int')
df['IloscMinus'] = df['IloscMinus'].astype('int')
df['KolejnyWDniu'] = df['KolejnyWDniu'].astype('int')
df['AsId'] = df['AsId'].astype('int')

df.info()

<class 'pandas.DataFrame'>
Index: 3693844 entries, 1106 to 3752610
Data columns (total 37 columns):
 #   Column              Dtype         
---  ------              -----         
 0   DokId               int64         
 1   SKU                 int64         
 2   TypPoz              int64         
 3   IloscPlus           int64         
 4   IloscMinus          int64         
 5   CenaPrzedRab        float64       
 6   CenaPoRab           float64       
 7   Wartosc             float64       
 8   CenaDet             float64       
 9   Data                datetime64[ns]
 10  KolejnyWDniu        int64         
 11  NrDok               str           
 12  TypDok              float64       
 13  Aktywny_x           float64       
 14  Razem               float64       
 15  DoZaplaty           float64       
 16  Zaplacono           float64       
 17  AsId                int64         
 18  Nazwa_towar         str           
 19  Kod                 float64       
 20  Opis1          

In [42]:
df.Producent.value_counts()

Producent
357.0    85195
340.0    76958
512.0    75085
337.0    60902
118.0    50802
         ...  
11.0         5
136.0        5
485.0        4
567.0        2
546.0        1
Name: count, Length: 332, dtype: int64

In [43]:
for col in df.columns:
    display(df[col].value_counts())

DokId
1982216    20347
1982218    19969
1842411    19611
1601036    15488
1216669    14536
           ...  
2211905        1
2211906        1
2211908        1
2215985        1
2221910        1
Name: count, Length: 828528, dtype: int64

SKU
38960    43148
49624    29820
96       24161
75       21708
49627    15970
         ...  
49954        1
25653        1
24070        1
50192        1
50191        1
Name: count, Length: 21865, dtype: int64

TypPoz
4    2987304
1     299076
0     220774
3      94575
6      77839
7      11526
2       2750
Name: count, dtype: int64

IloscPlus
1       2587316
0        385769
6         96572
2         96005
12        81244
         ...   
2000          1
228           1
212           1
475           1
295           1
Name: count, Length: 514, dtype: int64

IloscMinus
 0      3650829
 1         9333
 2         5538
 3         3795
 4         2800
         ...   
-61           1
 901          1
 453          1
 104          1
 170          1
Name: count, Length: 281, dtype: int64

...

Ruch
-0.000     2987921
-1.000        7328
-2.000        3312
 1.000        3188
-3.000        1883
            ...   
-10.760          1
-3.487           1
-1.231           1
-1.783           1
 3.959           1
Name: count, Length: 2484, dtype: int64

nazwa_dnia
Piątek          629000
Poniedziałek    628514
Sobota          594505
Środa           584111
Wtorek          575651
Czwartek        552230
Niedziela       129833
Name: count, dtype: int64

dlugi_weekend
False    3693844
Name: count, dtype: int64

niedziela_handlowa
False    3674266
True       19578
Name: count, dtype: int64

kategoria_dnia
Piątek                629000
Poniedziałek          628514
Sobota                594505
Środa                 584111
Wtorek                575651
Czwartek              552230
Niedziela             110255
Niedziela handlowa     19578
Name: count, dtype: int64